# Stage 4.04 — full serial run
Launch ID first. Launch OOD only after ID is 32/32 complete and the selected A100 is idle. `--resume` skips complete valid artifacts.

In [ ]:
import os,subprocess
from pathlib import Path
GPU="0"  # global physical GPU; change to one idle A100
R=Path.home()/"async-vla-latency-bench"; P=Path.home()/"LIBERO-plus"; N=Path.home()/"stage1-native"; OFT=Path.home()/"openvla-oft"; PY=Path.home()/"venv-stage4-openvla/bin/python"; OUT=Path.home()/"stage4"; MAN=OUT/"stage4_second_policy_manifest.csv"; SNAP=Path((OUT/"stage4_checkpoint_snapshot.txt").read_text().strip())
state=subprocess.run(["nvidia-smi","-i",GPU,"--query-gpu=memory.used,utilization.gpu","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.strip(); print("Physical GPU",GPU,"state:",state); used,util=[int(x.strip()) for x in state.split(',')]
if used>=500 or util>=5: raise SystemExit(f"STOP: physical GPU {GPU} is not idle: {state}")
base=os.environ.copy(); base.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONPATH":str(OFT)+os.pathsep+str(R)})
log=OUT/"stage4_id.log"; pidfile=OUT/"stage4_id.pid"; cmd=[str(PY),"-u","-m","async_vla_benchmark.scripts.run_stage4","--config",str(R/"async_vla_benchmark/configs/stage4.yaml"),"--manifest",str(MAN),"--output-dir",str(OUT),"--scene","id","--openvla-oft-checkout",str(OFT),"--checkpoint-snapshot",str(SNAP),"--resume","--verbose"]
fh=open(log,"ab"); proc=subprocess.Popen(cmd,cwd=R,env=base,stdout=fh,stderr=subprocess.STDOUT,start_new_session=True); pidfile.write_text(str(proc.pid)+"\n"); print("launched Stage 4 ID",proc.pid,log)

In [ ]:
import csv
def alive(pid):
    state=subprocess.run(["ps","-p",str(pid),"-o","stat="],capture_output=True,text=True).stdout.strip(); return bool(state) and not state.startswith("Z")
rows=list(csv.DictReader(open(OUT/"stage4_episode_results.csv"))) if (OUT/"stage4_episode_results.csv").exists() else []; ids={r['run_id'] for r in rows}; print(f"episodes {len(ids)}/64; remaining {64-len(ids)}")
for scene in ("id","ood"):
    path=OUT/f"stage4_{scene}.pid"; pid=int(path.read_text()) if path.exists() else -1; print(scene,"alive=",alive(pid),"pid=",pid); log=OUT/f"stage4_{scene}.log"; print("\n".join(log.read_text(errors="replace").splitlines()[-8:]) if log.exists() else "not started")

In [ ]:
rows=list(csv.DictReader(open(OUT/"stage4_episode_results.csv"))) if (OUT/"stage4_episode_results.csv").exists() else []; id_rows=[r for r in rows if r['scene_condition']=='id' and r.get('status','').startswith('ok')]; assert len({r['run_id'] for r in id_rows})==32,"STOP: ID must be 32/32 valid before OOD"
state=subprocess.run(["nvidia-smi","-i",GPU,"--query-gpu=memory.used,utilization.gpu","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.strip(); used,util=[int(x.strip()) for x in state.split(',')]
if used>=500 or util>=5: raise SystemExit(f"STOP: physical GPU {GPU} is not idle: {state}")
ood=os.environ.copy(); ood.update({"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg","PYTHONPATH":str(P)+os.pathsep+str(OFT)+os.pathsep+str(R),"MAGICK_HOME":str(N),"PATH":str(N/"bin")+os.pathsep+os.environ.get("PATH",""),"LD_LIBRARY_PATH":str(N/"lib")+os.pathsep+os.environ.get("LD_LIBRARY_PATH","")})
log=OUT/"stage4_ood.log"; pidfile=OUT/"stage4_ood.pid"; cmd=[str(PY),"-u","-m","async_vla_benchmark.scripts.run_stage4","--config",str(R/"async_vla_benchmark/configs/stage4.yaml"),"--manifest",str(MAN),"--output-dir",str(OUT),"--scene","ood","--openvla-oft-checkout",str(OFT),"--checkpoint-snapshot",str(SNAP),"--resume","--verbose"]
fh=open(log,"ab"); proc=subprocess.Popen(cmd,cwd=R,env=ood,stdout=fh,stderr=subprocess.STDOUT,start_new_session=True); pidfile.write_text(str(proc.pid)+"\n"); print("launched Stage 4 OOD",proc.pid,log)

In [ ]:
rows=list(csv.DictReader(open(OUT/"stage4_episode_results.csv"))) if (OUT/"stage4_episode_results.csv").exists() else []; ids={r['run_id'] for r in rows}; print(f"episodes {len(ids)}/64; remaining {64-len(ids)}")
for scene in ("id","ood"):
    path=OUT/f"stage4_{scene}.pid"; pid=int(path.read_text()) if path.exists() else -1; print(scene,"alive=",alive(pid),"pid=",pid); log=OUT/f"stage4_{scene}.log"; print("\n".join(log.read_text(errors="replace").splitlines()[-8:]) if log.exists() else "not started")
print("Checkpoint ~/stage4 off-machine regularly. Rerun the matching launch cell after interruption; --resume skips valid episodes.")